# 04 — Cleaning
**Goal:** Turn messy real-world data into a clean, reliable DataFrame.

In analytics, 80% of the work is cleaning. A bad clean means bad charts and bad decisions.

Topics:
- Missing values (`NaN`)
- Duplicates
- Renaming columns
- Type casting
- Outliers
- String cleaning

In [1]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np

# Simulate a messy Adobe Analytics + paid export
messy_data = {
    'Date':          ['01/01/2024','01/02/2024','01/02/2024','01/03/2024','01/04/2024','01/05/2024', None],
    'Channel':       ['Organic', 'PAID', 'paid', 'Email ', ' direct', 'Social', 'organic'],
    'Visits':        ['3,200', '2,800', '2,800', None, '4,100', '1,900', '3,100'],
    'Activations':   [96, 59, 59, 158, None, 47, 88],
    'Bounce Rate':   ['42.5%', '45.1%', '45.1%', '38.9%', '51.3%', '999%', '37.2%'],
    'Revenue ($)':   ['$4,800', '$2,950', '$2,950', '$7,900', '$2,350', '$-1,000', '$4,400'],
}

df = pd.DataFrame(messy_data)
print(df)
print()
print(df.dtypes)

         Date  Channel Visits  Activations Bounce Rate Revenue ($)
0  01/01/2024  Organic  3,200         96.0       42.5%      $4,800
1  01/02/2024     PAID  2,800         59.0       45.1%      $2,950
2  01/02/2024     paid  2,800         59.0       45.1%      $2,950
3  01/03/2024   Email     NaN        158.0       38.9%      $7,900
4  01/04/2024   direct  4,100          NaN       51.3%      $2,350
5  01/05/2024   Social  1,900         47.0        999%     $-1,000
6         NaN  organic  3,100         88.0       37.2%      $4,400

Date               str
Channel            str
Visits             str
Activations    float64
Bounce Rate        str
Revenue ($)        str
dtype: object


## 1. Inspect the mess first

In [2]:
# Count nulls per column
print('Nulls per column:')
print(df.isnull().sum())
print()

# Null percentage
print('Null percentage:')
print((df.isnull().sum() / len(df) * 100).round(1))

Nulls per column:
Date           1
Channel        0
Visits         1
Activations    1
Bounce Rate    0
Revenue ($)    0
dtype: int64

Null percentage:
Date           14.3
Channel         0.0
Visits         14.3
Activations    14.3
Bounce Rate     0.0
Revenue ($)     0.0
dtype: float64


In [3]:
# Check for duplicates
print('Duplicate rows:', df.duplicated().sum())
print()
print('Duplicate rows (detail):')
print(df[df.duplicated(keep=False)])   # keep=False shows all duplicates

Duplicate rows: 0

Duplicate rows (detail):
Empty DataFrame
Columns: [Date, Channel, Visits, Activations, Bounce Rate, Revenue ($)]
Index: []


## 2. Remove duplicates

In [4]:
# Drop exact duplicate rows — keep the first occurrence
df = df.drop_duplicates()
print('Shape after removing duplicates:', df.shape)

# Drop duplicates based on specific columns (Date + Channel should be unique)
# df = df.drop_duplicates(subset=['Date', 'Channel'], keep='first')

Shape after removing duplicates: (7, 6)


## 3. Handle missing values

In [5]:
# Option A: Drop rows with any null
# Use when nulls are rare and the row is meaningless without the value
df_dropped = df.dropna()                         # drop if ANY column is null
df_dropped = df.dropna(subset=['Date'])          # drop only if Date is null
print('Rows after dropna:', len(df_dropped))

# Option B: Fill nulls with a value
# Use when you can reasonably impute the missing value
df['Visits']      = df['Visits'].fillna('0')     # fill missing visits with '0'
df['Activations'] = df['Activations'].fillna(df['Activations'].median())  # fill with median

# Option C: Forward fill (carry last known value forward)
# Use for time series where a missing day inherits the previous day's value
df['Date'] = df['Date'].ffill()

print(df)

Rows after dropna: 6
         Date  Channel Visits  Activations Bounce Rate Revenue ($)
0  01/01/2024  Organic  3,200         96.0       42.5%      $4,800
1  01/02/2024     PAID  2,800         59.0       45.1%      $2,950
2  01/02/2024     paid  2,800         59.0       45.1%      $2,950
3  01/03/2024   Email       0        158.0       38.9%      $7,900
4  01/04/2024   direct  4,100         73.5       51.3%      $2,350
5  01/05/2024   Social  1,900         47.0        999%     $-1,000
6  01/05/2024  organic  3,100         88.0       37.2%      $4,400


## 4. Fix types and parse columns

In [6]:
# Parse dates
df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

# Fix Visits: '3,200' → 3200
df['Visits'] = df['Visits'].str.replace(',', '', regex=False).astype(int)

# Fix Bounce Rate: '42.5%' → 0.425
df['Bounce Rate'] = df['Bounce Rate'].str.replace('%', '', regex=False).astype(float) / 100

# Fix Revenue: '$4,800' → 4800.0
df['Revenue ($)'] = (
    df['Revenue ($)']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

print(df.dtypes)
print()
print(df)

Date           datetime64[us]
Channel                   str
Visits                  int64
Activations           float64
Bounce Rate           float64
Revenue ($)           float64
dtype: object

        Date  Channel  Visits  Activations  Bounce Rate  Revenue ($)
0 2024-01-01  Organic    3200         96.0        0.425       4800.0
1 2024-01-02     PAID    2800         59.0        0.451       2950.0
2 2024-01-02     paid    2800         59.0        0.451       2950.0
3 2024-01-03   Email        0        158.0        0.389       7900.0
4 2024-01-04   direct    4100         73.5        0.513       2350.0
5 2024-01-05   Social    1900         47.0        9.990      -1000.0
6 2024-01-05  organic    3100         88.0        0.372       4400.0


## 5. String cleaning — normalize channel names

In [7]:
# Before: 'Organic', 'PAID', 'paid', 'Email ', ' direct'
print('Before:', df['Channel'].unique())

df['Channel'] = (
    df['Channel']
    .str.strip()       # remove leading/trailing spaces
    .str.lower()       # normalize to lowercase
)

print('After: ', df['Channel'].unique())

Before: <ArrowStringArray>
['Organic', 'PAID', 'paid', 'Email ', ' direct', 'Social', 'organic']
Length: 7, dtype: str
After:  <ArrowStringArray>
['organic', 'paid', 'email', 'direct', 'social']
Length: 5, dtype: str


## 6. Rename columns — snake_case

In [8]:
df = df.rename(columns={
    'Date':        'date',
    'Channel':     'channel',
    'Visits':      'visits',
    'Activations': 'activations',
    'Bounce Rate': 'bounce_rate',
    'Revenue ($)': 'revenue',
})

print(df.columns.tolist())
print(df)

['date', 'channel', 'visits', 'activations', 'bounce_rate', 'revenue']
        date  channel  visits  activations  bounce_rate  revenue
0 2024-01-01  organic    3200         96.0        0.425   4800.0
1 2024-01-02     paid    2800         59.0        0.451   2950.0
2 2024-01-02     paid    2800         59.0        0.451   2950.0
3 2024-01-03    email       0        158.0        0.389   7900.0
4 2024-01-04   direct    4100         73.5        0.513   2350.0
5 2024-01-05   social    1900         47.0        9.990  -1000.0
6 2024-01-05  organic    3100         88.0        0.372   4400.0


## 7. Outliers — detect and handle

In [9]:
# Bounce rate > 1.0 is impossible (999% was in our data)
print('Bounce rate outliers:')
print(df[df['bounce_rate'] > 1.0])

# Revenue < 0 is also suspicious
print('Negative revenue:')
print(df[df['revenue'] < 0])

# Option A: drop the row
df = df[df['bounce_rate'] <= 1.0]

# Option B: cap at a max value
df['revenue'] = df['revenue'].clip(lower=0)   # floor at 0

print('\nClean data:')
print(df)

Bounce rate outliers:
        date channel  visits  activations  bounce_rate  revenue
5 2024-01-05  social    1900         47.0         9.99  -1000.0
Negative revenue:
        date channel  visits  activations  bounce_rate  revenue
5 2024-01-05  social    1900         47.0         9.99  -1000.0

Clean data:
        date  channel  visits  activations  bounce_rate  revenue
0 2024-01-01  organic    3200         96.0        0.425   4800.0
1 2024-01-02     paid    2800         59.0        0.451   2950.0
2 2024-01-02     paid    2800         59.0        0.451   2950.0
3 2024-01-03    email       0        158.0        0.389   7900.0
4 2024-01-04   direct    4100         73.5        0.513   2350.0
6 2024-01-05  organic    3100         88.0        0.372   4400.0


## 8. Full cleaning pipeline as a function
In a real project, put your cleaning logic in a function so it's reusable and testable.

In [10]:
def clean_adobe_export(path: str) -> pd.DataFrame:
    """Load and clean an Adobe Analytics CSV export."""
    df = pd.read_csv(path, skiprows=4, thousands=',')

    df = df.drop_duplicates()
    df = df.dropna(subset=['Date'])

    df['Date'] = pd.to_datetime(df['Date'], dayfirst=False)

    if 'Bounce Rate' in df.columns:
        df['Bounce Rate'] = df['Bounce Rate'].str.replace('%','',regex=False).astype(float) / 100
        df = df[df['Bounce Rate'] <= 1.0]

    if 'Revenue' in df.columns:
        df['Revenue'] = (
            df['Revenue'].str.replace('$','',regex=False)
                         .str.replace(',','',regex=False)
                         .astype(float)
                         .clip(lower=0)
        )

    df.columns = [c.lower().replace(' ','_').replace('(','').replace(')','') for c in df.columns]

    return df


# Use it
df_clean = clean_adobe_export('data/raw/samples/adobe_export.csv')
print(df_clean)
print()
print(df_clean.dtypes)

        date  visits  unique_visitors  page_views  bounce_rate  \
0 2024-01-01    3200             2800        9600        0.425   
1 2024-01-02    2800             2400        8400        0.451   
2 2024-01-03    3500             3100       10500        0.389   
3 2024-01-04    2600             2200        7800        0.513   
4 2024-01-05    4100             3600       12300        0.352   

   average_time_spent_seconds  orders  revenue  
0                         185      96   4800.0  
1                         172      59   2950.0  
2                         201     158   7900.0  
3                         154      47   2350.0  
4                         223     156   7800.0  

date                          datetime64[us]
visits                                 int64
unique_visitors                        int64
page_views                             int64
bounce_rate                          float64
average_time_spent_seconds             int64
orders                                

## Summary
| Task | How |
|---|---|
| Count nulls | `df.isnull().sum()` |
| Drop nulls | `df.dropna()` / `df.dropna(subset=['col'])` |
| Fill nulls | `df['col'].fillna(value)` |
| Forward fill | `df['col'].ffill()` |
| Drop duplicates | `df.drop_duplicates()` |
| Parse dates | `pd.to_datetime(df['col'])` |
| Strip spaces | `df['col'].str.strip()` |
| Lowercase | `df['col'].str.lower()` |
| Remove characters | `df['col'].str.replace('$','')` |
| Cap outliers | `df['col'].clip(lower=0, upper=100)` |
| Rename columns | `df.rename(columns={'old':'new'})` |

**Next:** `05_transforming.ipynb` — apply, map, assign, str and dt accessors.